In [1]:
###
# This script runs the simulations for fully synthetic data
# Here we generate one proxy network which is identitical to the true network
# We compare the results of BG sampler with the results of a sampler using the true network as a fixed variable (i.e., the "oracle" sampler)
###

# --- Import libraries ---

import jax.numpy as jnp
from jax import random, devices, local_device_count


# sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

import Simulations.data_gen as dg


from src.MWG_sampler import MWG_sampler, MWG_init
from src.MCMC_fixed_net import mcmc_fixed_net
import src.Models as models
import src.GWG as gwg
import pandas as pd

import numpyro


/Users/barwein/code/pystat/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- Set cores and seed ---


N_CORES = 4  # update accordingly, can use GPU as well
# os.environ["XLA_FLAGS"] = f"--xla_force_host_platform_device_count={N_CORES}"
numpyro.set_host_device_count(N_CORES)
devices("cpu")  # use CPU for parallelization, can switch to GPU if available
print("Number of cpu nodes:", local_device_count())  # check number of devices available

# --- Global variables ---

N = 500
TRIU_DIM = N * (N - 1) // 2

THETA = jnp.array([-2.5, 1])
# GAMMA_BASELINE = jnp.array([logit(0.95), logit(0.05)])
GAMMA_B_NOISE_0 = 15
GAMMA_B_NOISE_1 = -15
GAMMA = jnp.array([15, -15, 0])

ETA = jnp.array([-1, 3, -0.5, 2])
SIG_INV = 1.0
RHO = 0.5
PZ = 0.5

PARAM = {
    "theta": THETA,
    "eta": ETA,
    "rho": RHO,
    "gamma" : GAMMA,
    "sig_inv": SIG_INV,
}
FILEPATH = "Simulations/results"


N_ITER = 100

results = []


Number of cpu nodes: 4


In [3]:
# Aux function from simulation_aux.py 

def run_eval(mcmc_obj, model_name, new_interventions, true_vals, idx):
    # 1. Dynamic
    dyn_stats = mcmc_obj.new_intervention_error_stats(
        new_z=new_interventions.Z_h,
        true_estimands=new_interventions.estimand_h,
        true_vals=true_vals,
    )
    results.append(
        {
            "idx": idx,
            "model": model_name,
            "estimand": "dynamic",
            **dyn_stats,
        }
    )

    # 2. GATE aka TTE (Treat All vs None)
    gate_stats = mcmc_obj.new_intervention_error_stats(
        new_z=new_interventions.Z_gate,
        true_estimands=new_interventions.estimand_gate,
        true_vals=true_vals,
    )
    results.append(
        {
            "idx": idx,
            "model": model_name,
            "estimand": "gate",
            **gate_stats,
        }
    )

In [4]:
for i in range(N_ITER):

    print(f"### iteration {i} ###")
    # Set keys
    rng_key = random.PRNGKey(i)

    # generate data (not depedent on gamma)
    rng_key, _ = random.split(rng_key)
    fixed_data = dg.generate_fixed_data(rng_key, N, PARAM, PZ)

    # true_vals for wasserstein distance
    true_vals = {
        "eta": ETA,
        "rho": jnp.array([RHO]),
        "sig_inv": jnp.array([SIG_INV]),
        "triu_star": fixed_data["triu_star"],
    }

    print(f"mean true exposures: {jnp.mean(fixed_data['true_exposures'])}")

    # generate new interventions
    rng_key, _ = random.split(rng_key)
    new_interventions = dg.new_interventions_estimands(
        rng_key, N, fixed_data["x"], fixed_data["triu_star"], ETA
    )


    rng_key = random.split(rng_key)[0]
    # sample proxy networks with current gamma
    proxy_nets = dg.generate_proxy_networks(
        # rng,
        rng_key,
        TRIU_DIM,
        fixed_data["triu_star"],
        PARAM["gamma"],
        fixed_data["x_diff"],
        fixed_data["Z"],
    )

    data_sim = dg.data_for_sim(fixed_data, proxy_nets)

    # run one iteration
    rng_key = random.split(rng_key)[0]
    # a bit different in final results as it account for cluster job_id
    idx = str(i)

    print("--- fixed true net ---")
    rng_key = random.split(rng_key)[0]
    # --- fixed true network ---
    mcmc_true = mcmc_fixed_net(
        rng_key=rng_key,
        data=data_sim,
        net_type="true",
        num_chains=4,
        progress_bar=False,
    )
    run_eval(mcmc_obj = mcmc_true, 
            model_name = "true_net",
            new_interventions = new_interventions,
            true_vals = true_vals,
            idx = idx)

    # --- MWG sampler (single proxy) ---
    print("--- MWG init params (single proxy) ---")

    rng_key = random.split(rng_key)[1]

    mwg_init = MWG_init(
        rng_key=rng_key,
        data=data_sim,
        triu_star_grad_fn=models.triu_star_grad_fn,
        gwg_kernel_fn=gwg.GWG_kernel,
        n_iter_networks=20000,
        n_nets_samples=3000,
        gwg_init_steps=int(2e4),
        gwg_init_batch_len=5,
        refine_triu_star=True,
        progress_bar=False,
    ).get_init_values()

    print("--- Sampling with MWG (single proxy) ---")

    rng_key = random.split(rng_key)[1]

    mwg_sampler = MWG_sampler(
        rng_key=rng_key,
        data=data_sim,
        init_params=mwg_init,
        progress_bar=False,
        n_warmup=1000,
        n_samples=3000,
        gwg_n_steps=1,
        gwg_batch_len=1,
    )
    run_eval(mcmc_obj = mwg_sampler,
            model_name = "MWG",
            new_interventions = new_interventions,
            true_vals = true_vals,
            idx = idx)



### iteration 0 ###
mean true exposures: 1.3536434173583984
--- fixed true net ---
--- MWG init params (single proxy) ---
--- Sampling with MWG (single proxy) ---
### iteration 1 ###
mean true exposures: 1.8555872440338135
--- fixed true net ---
--- MWG init params (single proxy) ---
--- Sampling with MWG (single proxy) ---
### iteration 2 ###
mean true exposures: 1.826825737953186
--- fixed true net ---
--- MWG init params (single proxy) ---
--- Sampling with MWG (single proxy) ---
### iteration 3 ###
mean true exposures: 1.3178479671478271
--- fixed true net ---
--- MWG init params (single proxy) ---
--- Sampling with MWG (single proxy) ---
### iteration 4 ###
mean true exposures: 1.7871263027191162
--- fixed true net ---
--- MWG init params (single proxy) ---
--- Sampling with MWG (single proxy) ---
### iteration 5 ###
mean true exposures: 1.7213748693466187
--- fixed true net ---
--- MWG init params (single proxy) ---
--- Sampling with MWG (single proxy) ---
### iteration 6 ###
mea

In [5]:
# --- save results and write to csv ---
results_df = pd.DataFrame(results)

file_name = "Simulations/results/true_proxy_sim_results.csv"
results_df.to_csv(file_name, index=False, mode="a", header=True)
